# 跨模型互審 (Cross-Model Review)

## 模組脈絡：用「不同模型互相審查」抵銷單一模型盲點

本筆記是 **06-協作收斂** 的代表案例。單一模型有其偏誤與盲點；讓**不同廠商的模型互相審查**（一個寫、其他評），能抓出單一模型漏掉的問題——這是多模型協作把品質收斂的具體手法，也直接體現本課程「多模型並陳」的精神。

**架構**：Claude 撰寫 → GPT 與 Gemini 平行評審 → 綜合判斷 → 量化一致性。

## 0. 環境設定

In [ ]:
from dotenv import load_dotenv
import os, asyncio
load_dotenv()

from openai import OpenAI
import anthropic
from google import genai

openai_client = OpenAI()        # OPENAI_API_KEY
claude_client = anthropic.Anthropic()  # ANTHROPIC_API_KEY
gemini_client = genai.Client()  # GEMINI_API_KEY
import os
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")

## 1. Writer：用 Claude 產生初稿

In [ ]:
def write_draft(topic):
    msg = claude_client.messages.create(
        model="claude-sonnet-4-6", max_tokens=800,
        messages=[{"role": "user", "content": f"請寫一段約 200 字的技術說明：{topic}"}],
    )
    return msg.content[0].text

topic = "為什麼 RAG 能降低 LLM 幻覺"
draft = write_draft(topic)
print(draft)

## 2. Reviewers：GPT 與 Gemini 平行評審

兩個不同廠商的模型，各自從事實正確性、邏輯一致性、完整性評分並指出問題。

In [ ]:
REVIEW_PROMPT = (
    "你是嚴格的技術審稿人。請就以下文字評審，輸出 JSON："
    '{"score": 1-5, "issues": ["..."], "verdict": "accept/revise"}。'
    "評審面向：事實正確性、邏輯一致性、完整性。\n\n文字：\n"
)

def review_with_gpt(text):
    r = openai_client.responses.create(
        model=OPENAI_MODEL, text={"format": {"type": "json_object"}},
        input=[{"role": "user", "content": REVIEW_PROMPT + text}],
    )
    return r.output_text

def review_with_gemini(text):
    from google.genai import types
    r = gemini_client.models.generate_content(
        model="gemini-2.5-flash", contents=REVIEW_PROMPT + text,
        config=types.GenerateContentConfig(response_mime_type="application/json"),
    )
    return r.text

gpt_review = review_with_gpt(draft)
gemini_review = review_with_gemini(draft)
print("GPT 評審:", gpt_review)
print("Gemini 評審:", gemini_review)

## 3. 綜合：用第三方模型整合兩份評審

In [ ]:
def synthesize(draft, reviews):
    joined = "\n".join(f"- {r}" for r in reviews)
    msg = claude_client.messages.create(
        model="claude-sonnet-4-6", max_tokens=600,
        messages=[{"role": "user", "content": (
            f"原文：\n{draft}\n\n兩份評審：\n{joined}\n\n"
            "請綜合評審意見，指出兩位審稿人「都同意」與「有分歧」之處，並給出修改建議。"
        )}],
    )
    return msg.content[0].text

print(synthesize(draft, [gpt_review, gemini_review]))

## 4. 一致性量化（選做）

用 embedding 餘弦相似度衡量兩份評審的一致程度——高一致代表問題明確，低一致代表議題有爭議、值得人工複核。

In [ ]:
import numpy as np

def embed(text):
    return openai_client.embeddings.create(model="text-embedding-3-small", input=text).data[0].embedding

a, b = np.array(embed(gpt_review)), np.array(embed(gemini_review))
cos = float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
print(f"兩份評審一致性（餘弦相似度）: {cos:.3f}")

---

## 本章小結

1. **跨模型互審**：不同廠商模型互評，抵銷單一模型盲點。
2. 架構：Writer（Claude）→ 平行 Reviewers（GPT + Gemini）→ 綜合 → 一致性量化。
3. 低一致性的議題應觸發**人工複核**——把不確定收斂到人。
4. 這是「多模型並陳」最具體的協作收斂應用，呼應整個課程主線。